# Task 181: seismic_dvv — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Seismic velocity change estimation using stretching method

**Paper**: SeisMIC: Seismic velocity monitoring
**Repository**: https://github.com/PeterMakus/SeisMIC

### Physical Background

Seismic waves carry subsurface info. Inversion recovers velocity/impedance from recorded seismograms.

### Forward Model

```
d = G(m)  via wave equation propagation
```

### Inverse Problem

```
min ||d_obs - G(m)||^2 + lambda R(m)
```

### Performance Metrics
- **PSNR**: 22.12 dB (dv/v time series)
- **SSIM**: N/A (1D time series)
- **Evaluation**: 1D seismic velocity change estimation (dv/v CC, RE)

### Mathematical Formulation

Seismic wave propagation is governed by the wave equation:

$$\frac{\partial^2 u}{\partial t^2} = c^2(\mathbf{x}) \nabla^2 u + s(t)\,\delta(\mathbf{x} - \mathbf{x}_s)$$

**Full Waveform Inversion (FWI)**:
$$\hat{c} = \arg\min_c \sum_{s,r} \|u^{\text{obs}}_{s,r} - u^{\text{syn}}_{s,r}(c)\|_2^2$$

Gradient via adjoint method:
$$\frac{\partial J}{\partial c} = -\frac{2}{c^3} \int_0^T u_{\text{fwd}}(\mathbf{x},t) \cdot u_{\text{adj}}(\mathbf{x},t) \, dt$$

**Impedance inversion** from reflectivity: $r_i = \frac{Z_{i+1} - Z_i}{Z_{i+1} + Z_i}$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 181: Seismic dv/v Estimation via Stretching Method

Inverse Problem: Inverting seismic velocity change (dv/v) from ambient noise
cross-correlation using the stretching technique.

Forward: CCF_cur(t) = interp(CCF_ref, t*(1+eps)) where eps = -dv/v
Inverse: For trial eps values, stretch reference, compute CC with current CCF,
         find eps that maximizes CC => dv/v_est = -eps_best
"""

import json
import os
import numpy as np
from scipy.interpolate import interp1d

# ─────────────────────────── Parameters ───────────────────────────
SEED = 42
FS = 100.0            # sampling rate (Hz)
T_START, T_END = -10.0, 10.0
F0 = 2.0              # Ricker wavelet center frequency
N_DAYS = 50           # number of synthetic observation days
DVV_AMP = 0.002       # 0.2% amplitude of dv/v sinusoidal variation
DVV_NOISE = 0.0005    # 0.05% noise level
DVV_PERIOD = 30.0     # period of sinusoidal dv/v in days
NOISE_LEVEL = 0.02    # measurement noise added to perturbed CCFs
STRETCH_RANGE = 0.01  # ±1% trial stretching
STRETCH_STEPS = 201   # number of trial epsilon values
CODA_TMIN = 1.0       # coda window: |t| > CODA_TMIN seconds

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

rng = np.random.default_rng(SEED)


# ─────────────────────── Synthesise Reference CCF ──────────────────

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`make_ricker`, `main`

In [ ]:
# ─────────────────────── Synthesise Reference CCF ──────────────────
def make_ricker(t: np.ndarray, f0: float) -> np.ndarray:
    """Ricker (Mexican-hat) wavelet centred at t=0."""
    a = 1.0 / f0
    u = (np.pi * (t) / a) ** 2
    return (1.0 - 2.0 * u) * np.exp(-u)

# ──────────────────────── Main Pipeline ────────────────────────────
def main():
    # Time axis
    n_samples = int((T_END - T_START) * FS) + 1
    t = np.linspace(T_START, T_END, n_samples)

    # Step 1: Synthesise reference CCF
    ccf_ref = synthesise_reference_ccf(t, F0)
    print(f"Reference CCF: {len(ccf_ref)} samples, "
          f"t=[{t[0]:.1f}, {t[-1]:.1f}] s")

    # Step 2: Generate multi-day dataset with known dv/v
    days, dvv_true, ccf_matrix = synthesise_dataset(
        t, ccf_ref, N_DAYS, DVV_AMP, DVV_NOISE, DVV_PERIOD, NOISE_LEVEL)
    print(f"Synthesised {N_DAYS} days of CCF data")
    print(f"True dv/v range: [{dvv_true.min()*100:.4f}%, "
          f"{dvv_true.max()*100:.4f}%]")

    # Step 3: Invert dv/v using stretching method
    dvv_est = np.empty(N_DAYS)
    cc_best_arr = np.empty(N_DAYS)
    for d in range(N_DAYS):
        dvv_est[d], cc_best_arr[d], _ = inverse_stretching(
            ccf_ref, ccf_matrix[d], t,
            STRETCH_RANGE, STRETCH_STEPS, CODA_TMIN)

    print(f"Estimated dv/v range: [{dvv_est.min()*100:.4f}%, "
          f"{dvv_est.max()*100:.4f}%]")
    print(f"Mean best CC: {cc_best_arr.mean():.4f}")

    # Step 4: Evaluate
    metrics = compute_metrics(dvv_true, dvv_est)
    print("\n=== Metrics ===")
    for k, v in metrics.items():
        print(f"  {k}: {v:.6f}")

    # Save metrics
    metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"\nMetrics saved to {metrics_path}")

    # Step 5: Visualise
    plot_results(t, ccf_ref, ccf_matrix[10], days, dvv_true, dvv_est,
                 os.path.join(RESULTS_DIR, "reconstruction_result.png"))

    # Step 6: Save arrays
    np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), dvv_true)
    np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), dvv_est)
    print("Arrays saved.")

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
d = G(m)  via wave equation propagation
```

Functions: `synthesise_reference_ccf`, `forward_stretch`, `synthesise_dataset`

In [ ]:
def synthesise_reference_ccf(t: np.ndarray, f0: float,
                              decay_tau: float = 5.0,
                              n_scatterers: int = 12) -> np.ndarray:
    """
    Build a realistic-looking reference cross-correlation function.
    Ricker wavelet modulated by exponential decay + scattered arrivals.
    """
    ccf = make_ricker(t, f0) * np.exp(-np.abs(t) / decay_tau)
    # Add scattered arrivals (smaller Ricker wavelets at random times)
    for _ in range(n_scatterers):
        t_shift = rng.uniform(-8, 8)
        amp = rng.uniform(0.05, 0.25) * np.exp(-np.abs(t_shift) / decay_tau)
        f_scat = rng.uniform(1.5, 3.5)
        ccf += amp * make_ricker(t - t_shift, f_scat)
    return ccf

# ─────────────────────── Forward Operator ──────────────────────────
def forward_stretch(ccf_ref: np.ndarray, t: np.ndarray,
                    dvv: float) -> np.ndarray:
    """
    Forward operator: apply velocity change to reference CCF.
    CCF_cur(t) = CCF_ref(t * (1 + eps))  where eps = -dv/v
    """
    eps = -dvv
    t_stretched = t * (1.0 + eps)
    interp_func = interp1d(t, ccf_ref, kind='cubic',
                           bounds_error=False, fill_value=0.0)
    return interp_func(t_stretched)

# ──────────────────── Synthesise Multi-day Dataset ─────────────────
def synthesise_dataset(t, ccf_ref, n_days, dvv_amp, dvv_noise_std,
                       dvv_period, noise_level):
    """Create synthetic dv/v time series and perturbed CCFs."""
    days = np.arange(n_days)
    dvv_true = dvv_amp * np.sin(2.0 * np.pi * days / dvv_period) + \
        dvv_noise_std * rng.standard_normal(n_days)

    ccf_matrix = np.empty((n_days, len(t)))
    for d in range(n_days):
        ccf_cur = forward_stretch(ccf_ref, t, dvv_true[d])
        ccf_cur += noise_level * rng.standard_normal(len(t)) * np.max(
            np.abs(ccf_ref))
        ccf_matrix[d] = ccf_cur

    return days, dvv_true, ccf_matrix

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
min ||d_obs - G(m)||^2 + lambda R(m)
```

Functions: `inverse_stretching`

In [ ]:
# ─────────────────── Inverse Solver (Stretching) ───────────────────
def inverse_stretching(ccf_ref: np.ndarray, ccf_cur: np.ndarray,
                       t: np.ndarray, stretch_range: float,
                       stretch_steps: int,
                       coda_tmin: float) -> tuple:
    """
    Stretching method: find epsilon that maximises correlation between
    stretched reference and current CCF within the coda window.

    Returns (dv/v_est, cc_best, cc_curve)
    """
    # Coda window mask: |t| > coda_tmin
    coda_mask = np.abs(t) >= coda_tmin

    trial_eps = np.linspace(-stretch_range, stretch_range, stretch_steps)
    interp_func = interp1d(t, ccf_ref, kind='cubic',
                           bounds_error=False, fill_value=0.0)

    cc_values = np.empty(stretch_steps)
    cur_coda = ccf_cur[coda_mask]
    cur_coda_demean = cur_coda - cur_coda.mean()
    cur_norm = np.sqrt(np.sum(cur_coda_demean ** 2))

    for i, eps in enumerate(trial_eps):
        t_stretched = t * (1.0 + eps)
        ref_stretched = interp_func(t_stretched)
        ref_coda = ref_stretched[coda_mask]
        ref_coda_demean = ref_coda - ref_coda.mean()
        ref_norm = np.sqrt(np.sum(ref_coda_demean ** 2))
        if ref_norm < 1e-15 or cur_norm < 1e-15:
            cc_values[i] = 0.0
        else:
            cc_values[i] = np.sum(ref_coda_demean * cur_coda_demean) / (
                ref_norm * cur_norm)

    best_idx = np.argmax(cc_values)
    eps_best = trial_eps[best_idx]
    dvv_est = -eps_best
    return dvv_est, cc_values[best_idx], cc_values

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `plot_results`

In [ ]:
# ──────────────────────── Evaluation Metrics ───────────────────────
def compute_metrics(dvv_true: np.ndarray,
                    dvv_est: np.ndarray) -> dict:
    """Compute dv/v estimation quality metrics."""
    # Mean absolute error
    mae = float(np.mean(np.abs(dvv_est - dvv_true)))

    # Relative error (fraction of amplitude range)
    amp_range = np.max(dvv_true) - np.min(dvv_true)
    rel_error = float(mae / amp_range) if amp_range > 0 else float('inf')

    # Correlation coefficient
    cc = float(np.corrcoef(dvv_true, dvv_est)[0, 1])

    # PSNR (treating dv/v time series as 1D signal)
    mse = float(np.mean((dvv_est - dvv_true) ** 2))
    peak = float(np.max(np.abs(dvv_true)))
    if mse > 0 and peak > 0:
        psnr = float(20.0 * np.log10(peak / np.sqrt(mse)))
    else:
        psnr = float('inf')

    return {
        "dvv_mae": mae,
        "dvv_relative_error": rel_error,
        "dvv_correlation_coefficient": cc,
        "dvv_psnr_dB": psnr,
    }

# ──────────────────────── Visualisation ────────────────────────────
def plot_results(t, ccf_ref, ccf_cur_example, days, dvv_true, dvv_est,
                 save_path):
    """Multi-panel figure."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # (a) Reference CCF
    ax = axes[0, 0]
    ax.plot(t, ccf_ref, 'k', lw=0.8)
    ax.set_xlabel("Lag time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_title("(a) Reference CCF")
    ax.set_xlim(t[0], t[-1])

    # (b) Reference vs current (one example day)
    ax = axes[0, 1]
    ax.plot(t, ccf_ref, 'k', lw=0.8, label="Reference")
    ax.plot(t, ccf_cur_example, 'r', lw=0.8, alpha=0.7, label="Current (day 10)")
    ax.set_xlabel("Lag time (s)")
    ax.set_ylabel("Amplitude")
    ax.set_title("(b) Reference vs Perturbed CCF")
    ax.legend(fontsize=9)
    ax.set_xlim(t[0], t[-1])

    # (c) True vs estimated dv/v
    ax = axes[1, 0]
    ax.plot(days, dvv_true * 100, 'k-o', ms=3, lw=1.2, label="True dv/v")
    ax.plot(days, dvv_est * 100, 'r-s', ms=3, lw=1.2, label="Estimated dv/v")
    ax.set_xlabel("Day")
    ax.set_ylabel("dv/v (%)")
    ax.set_title("(c) dv/v Time Series")
    ax.legend(fontsize=9)

    # (d) Residual
    ax = axes[1, 1]
    residual = (dvv_true - dvv_est) * 100
    ax.bar(days, residual, color='steelblue', alpha=0.7)
    ax.axhline(0, color='k', ls='--', lw=0.5)
    ax.set_xlabel("Day")
    ax.set_ylabel("Residual dv/v (%)")
    ax.set_title("(d) Residual (True − Estimated)")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Figure saved to {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Time axis
n_samples = int((T_END - T_START) * FS) + 1
t = np.linspace(T_START, T_END, n_samples)

### Step 1: Synthesise reference CCF

Intermediate processing step in the pipeline.

In [ ]:
# Step 1: Synthesise reference CCF
ccf_ref = synthesise_reference_ccf(t, F0)
print(f"Reference CCF: {len(ccf_ref)} samples, "
      f"t=[{t[0]:.1f}, {t[-1]:.1f}] s")

### Step 2: Generate multi-day dataset with known dv/v

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 2: Generate multi-day dataset with known dv/v
days, dvv_true, ccf_matrix = synthesise_dataset(
    t, ccf_ref, N_DAYS, DVV_AMP, DVV_NOISE, DVV_PERIOD, NOISE_LEVEL)
print(f"Synthesised {N_DAYS} days of CCF data")
print(f"True dv/v range: [{dvv_true.min()*100:.4f}%, "
      f"{dvv_true.max()*100:.4f}%]")

### Step 3: Invert dv/v using stretching method

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 3: Invert dv/v using stretching method
dvv_est = np.empty(N_DAYS)
cc_best_arr = np.empty(N_DAYS)
for d in range(N_DAYS):
    dvv_est[d], cc_best_arr[d], _ = inverse_stretching(
        ccf_ref, ccf_matrix[d], t,
        STRETCH_RANGE, STRETCH_STEPS, CODA_TMIN)

print(f"Estimated dv/v range: [{dvv_est.min()*100:.4f}%, "
      f"{dvv_est.max()*100:.4f}%]")
print(f"Mean best CC: {cc_best_arr.mean():.4f}")

### Step 4: Evaluate

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 4: Evaluate
metrics = compute_metrics(dvv_true, dvv_est)
print("\n=== Metrics ===")
for k, v in metrics.items():
    print(f"  {k}: {v:.6f}")

# Save metrics
metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nMetrics saved to {metrics_path}")

### Step 5: Visualise

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 5: Visualise
plot_results(t, ccf_ref, ccf_matrix[10], days, dvv_true, dvv_est,
             os.path.join(RESULTS_DIR, "reconstruction_result.png"))

### Step 6: Save arrays

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 6: Save arrays
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), dvv_true)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), dvv_est)
print("Arrays saved.")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **seismic_dvv**:

1. **Problem Setup**: Seismic waves carry subsurface info. Inversion recovers velocity/impedance from recorded seismograms.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=22.12 dB (dv/v time series), SSIM=N/A (1D time series))

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: SeisMIC: Seismic velocity monitoring
- Repository: https://github.com/PeterMakus/SeisMIC
